In [24]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
ham_dir = "data/ham/" 
spam_dir = "data/spam/"

In [ ]:
def extract_file_names(spam_dir: str, ham_dir: str):
    spam_file_names = None 
    non_spam_file_names = None 
    for (root, dirs, files) in os.walk(ham_dir, topdown=True):
        non_spam_file_names = files

    for (root, dirs, files) in os.walk(spam_dir, topdown=True):
        spam_file_names = files
        
    return spam_file_names, non_spam_file_names

def extract_file_raw(file_name: str, content: list):
    file = open(file_name, 'r', errors='ignore')
    content.append(file.read())
    return content



In [27]:
def extract_file_processed(ham_dir: str, spam_dir: str,lowercase= True,
                           punctuation= True, stemming= False,
                           convert_to_generic= False):

    spam_file_content = []
    ham_file_content = []
    spam_file_names, ham_file_names = extract_file_names(spam_dir, ham_dir)

    for i in range(len(spam_file_names)):
        extract_file_raw(os.path.join(spam_dir, spam_file_names[i]), spam_file_content )
    for i in range(len(ham_file_names)):
        extract_file_raw(os.path.join(ham_dir, ham_file_names[i]), ham_file_content)

    return spam_file_content, ham_file_content

In [28]:
spam_content, ham_content = extract_file_processed(ham_dir=ham_dir, spam_dir=spam_dir)
X_raw = spam_content + ham_content
y = np.array([1] * len(spam_content) + [0] * len(ham_content)) 

In [29]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42 )

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import CountVectorizer


knn_params = {
    'knn__n_neighbors': [3, 4, 5, 10],
    'knn__weights' : ['uniform', 'distance'],
}

knn_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(stop_words="english",decode_error='ignore',min_df=2)),
    ('knn',KNeighborsClassifier()) 
])
cv_strat =StratifiedKFold(n_splits=10, shuffle=True, random_state=45)

knn_grid_search = GridSearchCV(estimator=knn_pipeline, param_grid=knn_params, scoring='roc_auc_ovr', cv=cv_strat)
knn_grid_search.fit(X_train_raw,y_train)



,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'knn__n_neighbors': [3, 4, ...], 'knn__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc_ovr'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose

In [31]:
print("Best parameters found:", knn_grid_search.best_knn_params_)
print("Best CV ROC-AUC score:", knn_grid_search.best_score_)

Best parameters found: {'knn__n_neighbors': 10, 'knn__weights': 'distance'}
Best CV ROC-AUC score: 0.9959340597710055


In [32]:
from sklearn.metrics import classification_report, roc_auc_score

# Evaluate on the true holdout test set using the pipeline's best estimator
best_model = knn_grid_search.best_estimator_

# Get predictions
y_pred = best_model.predict(X_test_raw)
y_proba = best_model.predict_proba(X_test_raw)[:, 1]

print("Test ROC-AUC Score:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test ROC-AUC Score: 0.9928518479296834

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       497
           1       0.98      0.82      0.90       114

    accuracy                           0.96       611
   macro avg       0.97      0.91      0.94       611
weighted avg       0.96      0.96      0.96       611



In [35]:
from sklearn.linear_model import LogisticRegression

reg_params= {
        'reg__C': [ 0.1, 1.0 , 10.0]
}

reg_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(stop_words="english",decode_error='ignore',min_df=2)),
    ('reg',LogisticRegression()) 
])
cv_strat =StratifiedKFold(n_splits=10, shuffle=True, random_state=45)

reg_grid_search = GridSearchCV(estimator=reg_pipeline, param_grid=reg_params, scoring='roc_auc_ovr', cv=cv_strat)
reg_grid_search.fit(X_train_raw,y_train)



,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'reg__C': [0.1, 1.0, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc_ovr'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of infor

In [36]:
print("Best parameters found:", reg_grid_search.best_params_)
print("Best CV ROC-AUC score:", reg_grid_search.best_score_)

Best parameters found: {'reg__C': 10.0}
Best CV ROC-AUC score: 0.9988930246069098


In [37]:
best_reg_model = reg_grid_search.best_estimator_

y_pred = best_reg_model.predict(X_test_raw)
y_proba = best_reg_model.predict_proba(X_test_raw)[:,1]

print("\n--- Test Set Performance---")
print("Test ROC-AUC Score:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


--- Test Set Performance---
Test ROC-AUC Score: 0.99989410145081

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       497
           1       0.98      0.98      0.98       114

    accuracy                           0.99       611
   macro avg       0.99      0.99      0.99       611
weighted avg       0.99      0.99      0.99       611

